In [ ]:
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

import aeon.io.api as api
from aeon.io import reader, video
from aeon.schema.dataset import exp02
from aeon.analysis.utils import visits

In [ ]:
# Get sessions
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

root = Path("Y:/aeon/data/raw/AEON3/experiment0.3")
if not root.exists():
    print("Cannot find root paths. Check path names or connection.")
subject_events = api.load(root, exp02.ExperimentalMetadata.SubjectState)
sessions = visits(subject_events[subject_events.id.str.startswith("BAA")])
sessions

In [ ]:
# Prettify sessions

pd.options.mode.chained_assignment = None  # turn off "SettingWithCopy" warning for this cell

sessions = sessions[sessions.enter.dt.date >= date(2022, 10, 12)]
sessions = sessions[sessions.enter.dt.date <= date(2022, 10, 14)]
sessions = sessions[sessions.duration > pd.Timedelta("1 hour")]
sessions.loc[:, ("weight_enter")] = sessions["weight_enter"].astype(float).round(1)
sessions.loc[:, ("weight_exit")] = sessions["weight_exit"].astype(float).round(1)
sessions.loc[:, ("enter")] = sessions["enter"].dt.floor("1s")
sessions.loc[:, ("exit")] = sessions["exit"].dt.ceil("1s")
sessions.loc[:, ("duration")] = sessions["duration"].round("1s")
sessions = sessions[["id", "enter", "exit", "duration", "weight_enter", "weight_exit"]]
sessions = sessions.sort_values(by="enter")
sessions = sessions.reset_index()
sessions = sessions.drop(columns=["index"])
pd.options.mode.chained_assignment = "warn"
display(sessions)

In [ ]:
for s in sessions.itertuples():
    harp_reader = reader.Harp(pattern="Patch1_35*", columns=["TriggerPellet"])
    new_pellet_trig_bitmask = api.load(root, harp_reader, start=s.enter, end=s.exit).iloc[0, 0]
    new_pellet_trig_reader_p1 = reader.BitmaskEvent("Patch1_35*", new_pellet_trig_bitmask, "TriggerPellet")
    new_pellet_trig_reader_p2 = reader.BitmaskEvent("Patch2_35*", new_pellet_trig_bitmask, "TriggerPellet")
    p1 = api.load(root, new_pellet_trig_reader_p1, start=s.enter, end=s.exit)
    p2 = api.load(root, new_pellet_trig_reader_p2, start=s.enter, end=s.exit)

    # Use the patch where pellet dispension was triggered the most
    if len(p1) > len(p2):
        p = p1
    else:
        p = p2

    # Get start and end times of bouts of pellet dispension
    times = p.index.tolist()
    start_times = [times[0]]
    end_times = []
    for i in range(1,len(times)):
        if times[i] - times[i-1] > pd.Timedelta("5 minutes"):
            end_times.append(times[i-1])
            start_times.append(times[i])
    # Drop the last start time
    start_times = start_times[:-1]

    # Remove a minute from each of the start times
    start_times = [t - pd.Timedelta("1 minute") for t in start_times]
    # Add a minute to each of the end times
    end_times = [t + pd.Timedelta("1 minute") for t in end_times]

    # Directory to export videos to
    vid_export_dir = "Y:/aeon/code/scratchpad/sleap/multi_point_tracking/single_animal_CameraPatch/"

    for i in range(len(start_times)):
        # Export video
        frames_info = api.load(root, exp02.CameraPatch1.Video, start=start_times[i], end=end_times[i])
        vid = video.frames(frames_info)
        save_path = vid_export_dir + f"CameraPatch1_foraging_{i}.avi"
        video.export(vid, save_path, fps=125)